# Apéndice · 3 de 4 — Los modelos del apéndice, medidos como en la 08

La [notebook 2](lichess_2_entrenamiento.ipynb) dejó cuatro modelos entrenados con
5, 10 y 30 millones de posiciones del dump de evaluaciones de Lichess, y una
curva de RMSE que baja un 7 % cada vez que se duplican los datos. **Bajar el RMSE
no es el objetivo del trabajo**: el objetivo es un motor que juegue. Esta
notebook les hace a esos cuatro modelos **las mismas mediciones que la
[08](../08_motor_y_partidas.ipynb)** les hizo a los dos de la memoria —el mismo
test, las mismas 400 posiciones, las mismas aperturas, la misma escalera— y al
final los compara.

| sección | qué mide | en la 08 |
|---|---|---|
| 4 | RMSE y desglose sobre el test del proyecto | secciones 5.1 a 5.3 |
| 5 | tiempo por jugada (requerimiento 1.7) | sección 6 |
| 6 | pérdida media en centipeones y posiciones de referencia | secciones 7 y 7.2 |
| 7 | escalera de Elo a 1 y 2 plies | sección 8, con un escalón más |
| 8 | contra Stockfish a un ply | sección 9 |
| 9 | **comparación con los modelos de la memoria** | — |

**Los modelos de la memoria no se vuelven a medir.** Sus números se toman de la
corrida de la 08 y están escritos en la sección 9, con su origen. Volver a
jugarlos daría resultados apenas distintos —Stockfish con límite de tiempo no es
determinista— sin agregar información. La única excepción es un control de un
minuto: el RMSE de la ResNet de la memoria tiene que dar acá el 0,2511 de la 08,
que es lo que garantiza que el test es el mismo.

**Por qué el test del proyecto.** Los modelos del apéndice se entrenaron con otras
etiquetas: otro Stockfish, otra profundidad, 13 % de mates contra 1,4 %. Su RMSE
sobre el test de Lichess (notebook 2) no se compara con el de la memoria. Sobre
el test del proyecto —el mismo split y las mismas etiquetas que midió la 08— es
una medición de **transferencia**: la única vara común, y la que corresponde,
porque el motor juega en ese dominio.

**Un escalón más en la escalera.** La de la 08 llega a 1700, y los modelos del
apéndice juegan más fuerte: en una corrida preliminar el mejor sacó 0,67 puntos
por partida contra ese escalón. Acá se agrega **1900**, para que también tengan
rivales por encima. El Elo se reporta ajustado contra los cuatro escalones y,
aparte, contra los tres de la 08: la diferencia entre los dos dice cuánto mueve
el número el escalón nuevo.

> **Runtime de GPU, alrededor de dos horas.** La escalera son 8 motores × 4
> escalones × 30 partidas; la celda de la sección 7 estima el costo antes de
> lanzar nada.

## 1. Entorno

In [ ]:
# Clonar el repositorio e instalar el paquete.
# Idempotente: se puede volver a correr tal cual despues de una desconexion.
import os, sys, subprocess, importlib
from pathlib import Path

REPO_DIR = Path("/content/CEIA-TF-Chess-DL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/zFonta/CEIA-TF-Chess-DL.git", str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

# pip registra el paquete editable con un .pth, y los .pth solo se procesan al
# arrancar el interprete: un kernel que ya esta corriendo no lo ve. Agregar src/
# a sys.path lo hace visible sin tener que reiniciar el runtime.
src = str(REPO_DIR / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

# Stockfish con version fija. Entrenar no lo usa, pero sin el se saltean los 17
# tests de integracion del pipeline, que son la evidencia del requerimiento 3.2.
subprocess.run(["bash", "scripts/setup_stockfish.sh"], check=True)
os.environ["PATH"] = f"{REPO_DIR}/bin:" + os.environ["PATH"]

print("Listo. Directorio de trabajo:", os.getcwd())

In [ ]:
import numpy as np
import torch
from chessdl.colab import TRAINING, describe_runtime

runtime = describe_runtime(phase=TRAINING)
print("GPU  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")
for aviso in runtime.warnings():
    print("AVISO:", aviso)

## 2. Dataset y partición

El dataset del proyecto con su partición por partida, exactamente como en la 08.
Los modelos del apéndice nunca vieron estas posiciones: se entrenaron con otra
fuente.

In [ ]:
from chessdl import hf
from chessdl.config import load_config
from chessdl.data import schema
from chessdl.training.cache import build_cache, cache_path_for, load_cache
from chessdl.training.split import leaked_games, split_masks

cfg = load_config()
token = hf.get_token()
directorio = hf.download_dataset(cfg.output.hf_repo_id, "/content/ceia-chess/hub")
tabla = schema.read_dataset(schema.shard_paths(directorio))

fens     = tabla["fen"].to_pylist()
game_ids = tabla["game_id"].to_pylist()
plies    = np.asarray(tabla["ply"])
controles = np.asarray(tabla["time_control"])
targets  = np.asarray(tabla["value_stm"], dtype=np.float32)

masks = split_masks(game_ids, cfg.training.split_config())
assert not leaked_games(masks, game_ids)
idx_test = np.flatnonzero(masks['test'])
idx_val  = np.flatnonzero(masks['val'])

ruta_cache = cache_path_for(cfg.training.cache_dir)
if not ruta_cache.exists():
    build_cache(fens, ruta_cache, progress=True)
cache = load_cache(ruta_cache, expected_rows=len(fens))

print(f"val:  {len(idx_val):,} posiciones")
print(f"test: {len(idx_test):,} posiciones")

## 3. Los cuatro modelos del apéndice

Los mejores checkpoints por validación de cada brazo de la notebook 2. La ResNet
de la memoria se carga **sólo como control** del test (sección 4.2): no juega.

In [ ]:
from chessdl.engine.evaluator import Evaluator
from chessdl.engine.loader import load_from_hub

repo_modelos = f'{cfg.output.hf_namespace}/{cfg.training.hf_models_repo}'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

CORRIDAS = {
    '5M-chica':   'pruebas-lichess/5M-chica',
    '10M-chica':  'pruebas-lichess/10M-chica',
    '30M-chica':  'pruebas-lichess/30M-chica',
    '30M-grande': 'pruebas-lichess/30M-grande',
}

evaluadores = {}
for nombre, corrida in CORRIDAS.items():
    modelo = load_from_hub(repo_modelos, corrida, token=token, device=dispositivo)
    evaluadores[nombre] = Evaluator(modelo, device=dispositivo)
    print(f'{nombre:<12}{evaluadores[nombre].describe()}')

# Solo para el control de la seccion 4.2: no entra en ninguna otra medicion.
control = Evaluator(load_from_hub(repo_modelos, 'campana2-warmup', token=token,
                                  device=dispositivo), device=dispositivo)

## 4. Métricas sobre el test del proyecto

### 4.1 Tiempo de inferencia por lote

In [ ]:
import time
from chessdl.training.cache import cached_to_tensor

muestra = torch.from_numpy(cached_to_tensor(np.asarray(cache[idx_test[:4096]]))).to(dispositivo)

print(f"{'arquitectura':<14}{'lote':>7}{'ms':>10}{'pos/s':>12}")
print('-' * 43)
rendimiento = {}
for nombre, ev in evaluadores.items():
    for n in (32, 256, 2048):
        lote = muestra[:n]
        with torch.no_grad():
            for _ in range(3):
                ev.model(lote)                      # calentamiento
            if dispositivo == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(10):
                ev.model(lote)
            if dispositivo == 'cuda':
                torch.cuda.synchronize()
            dt = (time.perf_counter() - t0) / 10
        print(f"{nombre if n == 32 else '':<14}{n:>7}{dt*1000:>10.1f}{n/dt:>12,.0f}")
        rendimiento[(nombre, n)] = n / dt

### 4.2 RMSE, desglose por control de tiempo y requerimiento 1.4

La tabla principal del apéndice: los cuatro modelos sobre el test del proyecto,
con la misma regla que dio el 0,2511 y el 0,2531 de la memoria. Antes de leerla,
el **control**: la ResNet de la memoria tiene que reproducir su 0,2511 sobre
estas posiciones. Si no lo hace, el test no es el mismo de la 08 y nada de lo
que sigue se compara.

In [ ]:
from chessdl.training.loop import predict
from chessdl.training.metrics import evaluate

orden = np.sort(idx_test)
orden_val = np.sort(idx_val)

# Control: el mismo test que la 08, o nada de lo que sigue se compara.
RMSE_08 = {'ResNet memoria': 0.2511, 'Transformer memoria': 0.2531}
rmse_control = evaluate(predict(control.model, cache, targets, orden, dispositivo),
                        targets[orden]).rmse
assert abs(rmse_control - RMSE_08['ResNet memoria']) < 0.002, (
    f'la ResNet de la memoria da {rmse_control:.4f} y en la 08 dio 0.2511: '
    'el test o el cache no son los mismos')
print(f'control: la ResNet de la memoria da {rmse_control:.4f} (0.2511 en la 08)  [OK]\n')

predicciones, metricas = {}, {}
for nombre, ev in evaluadores.items():
    predicciones[nombre] = predict(ev.model, cache, targets, orden, dispositivo)
    pred_val = predict(ev.model, cache, targets, orden_val, dispositivo)
    metricas[nombre] = (evaluate(pred_val, targets[orden_val]),
                        evaluate(predicciones[nombre], targets[orden]))

print(f"{'modelo':<14}{'RMSE val':>10}{'RMSE test':>11}{'vs 0.2511':>11}"
      f"{'MAE cp':>9}{'signo':>9}{'rho':>8}")
print('-' * 72)
for nombre, (v, t) in metricas.items():
    print(f'{nombre:<14}{v.rmse:>10.4f}{t.rmse:>11.4f}'
          f'{t.rmse / RMSE_08["ResNet memoria"] - 1:>+11.1%}'
          f'{t.mae_cp:>9.1f}{t.sign_agreement:>9.1%}{t.spearman:>8.4f}')

print()
print(f"{'control':<14}{'posiciones':>12}", end='')
for n in evaluadores: print(f'{n:>12}', end='')
print()
print('-' * (26 + 12 * len(evaluadores)))
for control_t in ['Blitz', 'Rapid', 'Classical']:
    sel = controles[orden] == control_t
    if sel.sum() == 0:
        continue
    print(f'{control_t:<14}{sel.sum():>12,}', end='')
    for nombre, pred in predicciones.items():
        print(f'{evaluate(pred[sel], targets[orden][sel]).rmse:>12.4f}', end='')
    print()

# Requerimiento 1.4: el 100 % de la salida sobre el test completo, en [-1, 1].
print()
print('REQUERIMIENTO 1.4: LA SALIDA SOBRE TODO EL SPLIT DE TEST')
for nombre, pred in predicciones.items():
    dentro = float(np.mean(np.abs(pred) <= 1.0))
    print(f'  {nombre:<12}{len(pred):>9,} posiciones   rango [{pred.min():+.4f}, '
          f'{pred.max():+.4f}]   dentro de [-1, 1]: {dentro:.1%}')
    assert dentro == 1.0, f'{nombre}: salida fuera de [-1, 1]'

### 4.3 Según qué tan pareja está la posición

Como en la sección 5.3 de la 08: el acuerdo de signo por franja de |valor|, que
es lo que predice cómo elige el motor —la búsqueda decide casi siempre entre
posiciones cercanas en valor—. Se agrega el RMSE por franja, que dice **dónde**
compraron la mejora los datos extra: en las posiciones parejas, que son las que
deciden las jugadas, o en las ya decididas, que es donde el dump de Lichess
tiene muchas más (13 % de mates).

In [ ]:
from chessdl.training.metrics import sign_agreement

cortes = [(0.00, 0.05, 'casi igualada'), (0.05, 0.20, 'ligera ventaja'),
          (0.20, 0.50, 'ventaja clara'), (0.50, 1.01, 'decidida')]

print(f"{'franja':<18}{'|valor|':>12}{'posiciones':>12}", end='')
for n in evaluadores: print(f"{n:>12}", end='')
print()
print('ACUERDO DE SIGNO')
print('-' * (42 + 12 * len(evaluadores)))
for bajo, alto, etiqueta in cortes:
    reales = targets[orden]
    sel = (np.abs(reales) >= bajo) & (np.abs(reales) < alto)
    print(f"{etiqueta:<18}{f'{bajo:.2f}-{alto:.2f}':>12}{sel.sum():>12,}", end='')
    for nombre, pred in predicciones.items():
        # deadband=0: `sign_agreement` excluye por defecto las posiciones con
        # |valor| <= 0,05, que es *exactamente* la primera franja -- la mas
        # interesante de todas. Con el valor por defecto esa fila vuelve `nan`.
        print(f"{sign_agreement(pred[sel], reales[sel], deadband=0.0):>12.1%}", end='')
    print()

print()
print(f"{'franja':<18}{'|valor|':>12}{'posiciones':>12}", end='')
for n in evaluadores: print(f"{n:>12}", end='')
print()
print('RMSE')
print('-' * (42 + 12 * len(evaluadores)))
for bajo, alto, etiqueta in cortes:
    reales = targets[orden]
    sel = (np.abs(reales) >= bajo) & (np.abs(reales) < alto)
    print(f"{etiqueta:<18}{f'{bajo:.2f}-{alto:.2f}':>12}{sel.sum():>12,}", end='')
    for nombre, pred in predicciones.items():
        print(f"{evaluate(pred[sel], reales[sel]).rmse:>12.4f}", end='')
    print()

## 5. Requerimiento 1.7: tiempo por jugada según la profundidad

La misma medición de la sección 6 de la 08: percentil 99 sobre 100 posiciones de
test repartidas por el split, contra el presupuesto de 5 segundos. Importa sobre
todo para el **30M-grande**, que tiene casi el doble de parámetros: si no entrara
a 2 plies, no podría jugar la escalera a esa profundidad.

In [ ]:
import chess
from chessdl.engine.search import search

PRESUPUESTO = 5.0        # segundos, requerimiento 1.7
POSICIONES_1_7 = 100     # el plan pide al menos 100, con distinto numero de jugadas legales
POSICIONES_PROF3 = 8     # la 3 ya se sabe que no entra; alcanza para mostrarlo

# Repartidas a lo largo del test, no las primeras: las de una misma partida
# estan juntas en `orden`, y cien seguidas serian veinticinco partidas.
paso = max(1, len(orden) // (2 * POSICIONES_1_7))
prueba = [f for f in (fens[i] for i in orden[::paso])
          if not chess.Board(f).is_game_over()][:POSICIONES_1_7]
legales = [chess.Board(f).legal_moves.count() for f in prueba]
print(f'{len(prueba)} posiciones de test, entre {min(legales)} y {max(legales)} jugadas legales '
      f'(mediana {int(np.median(legales))})\n')

print(f"{'arquitectura':<14}{'prof':>6}{'pos':>6}{'hojas':>10}{'mediana ms':>12}"
      f"{'p99 ms':>10}{'max ms':>10}{'margen':>9}")
print('-' * 77)
factibles, costo, tiempos_1_7 = {}, {}, {}
for nombre, ev in evaluadores.items():
    factibles[nombre] = []
    for profundidad in (1, 2, 3):
        tiempos, hojas = [], []
        for fen in prueba[:POSICIONES_PROF3 if profundidad == 3 else POSICIONES_1_7]:
            r = search(chess.Board(fen), ev, depth=profundidad)
            tiempos.append(r.seconds); hojas.append(r.leaves)
        t = np.array(tiempos) * 1000
        p99 = float(np.percentile(t, 99))
        entra = p99 / 1000 < PRESUPUESTO
        if entra:
            factibles[nombre].append(profundidad)
        costo[(nombre, profundidad)] = float(np.median(t)) / 1000
        tiempos_1_7[(nombre, profundidad)] = t
        print(f"{nombre if profundidad == 1 else '':<14}{profundidad:>6}{len(t):>6}"
              f"{int(np.mean(hojas)):>10,}{np.median(t):>12.0f}{p99:>10.0f}{t.max():>10.0f}"
              f"{(PRESUPUESTO*1000/p99):>8.0f}x" + ('' if entra else '   NO ENTRA'))

print()
for nombre, ds in factibles.items():
    print(f'{nombre}: profundidades dentro del presupuesto -> {ds or [1]}')
    factibles[nombre] = ds or [1]

## 6. Calidad de jugada: pérdida media en centipeones

Las **mismas 400 posiciones** de la sección 7 de la 08 —misma semilla sobre el
mismo test—, contra Stockfish a profundidad 12. Es la medición mejor resuelta de
la fuerza, y la que la 08 usó para comparar motores parecidos.

In [ ]:
import chess.engine
from chessdl.engine.match import move_quality

POSICIONES_ACPL = 400        # subir si sobra tiempo; el intervalo se angosta con la raiz
PROF_REFERENCIA = 12         # la misma con la que se etiqueto el dataset

rng = np.random.default_rng(0)
muestra_acpl = [fens[i] for i in rng.choice(orden, size=POSICIONES_ACPL, replace=False)]
limite_sf = chess.engine.Limit(depth=PROF_REFERENCIA)

calidades = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    for nombre, ev in evaluadores.items():
        for profundidad in factibles[nombre]:
            clave = f'{nombre} d{profundidad}'
            calidades[clave] = move_quality(
                ev, sf, muestra_acpl, limite_sf,
                depth=profundidad, label=clave, progress=True,
            )
            print(calidades[clave].summary()); print()

In [ ]:
print(f"{'motor':<20}{'ACPL':>10}{'mediana':>10}{'acuerdo':>10}{'>300cp':>9}")
print('-' * 59)
for clave, q in sorted(calidades.items(), key=lambda kv: kv[1].acpl):
    print(f"{clave:<20}{q.acpl:>10.1f}{np.median(q.losses_cp):>10.1f}"
          f"{q.agreement:>10.1%}{q.blunder_rate():>9.1%}")

### 6.2 Posiciones de referencia

Las doce posiciones de la sección 7.2 de la 08, con la red evaluando cada una tal
cual, sin búsqueda. Interesa en particular el **mate en 1**: el dump de Lichess
tiene diez veces más mates que el dataset del proyecto, así que es de esperar que
estos modelos los lean más cerca de ±1. Y la **dama en prise**, la posición donde
una evaluación estática más fácilmente se equivoca.

In [ ]:
from chessdl.engine.search import value_white
from chessdl.normalize import score_to_cp, value_to_cp

REFERENCIA = [
    ('Posicion inicial',                'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'),
    ('Apertura italiana',               'r1bqk1nr/pppp1ppp/2n5/2b1p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4'),
    ('Final de peones simetrico',       '8/pp3kpp/8/8/8/8/PP3KPP/8 w - - 0 1'),
    ('Blancas con un caballo de menos', 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/R1BQKBNR w KQkq - 0 1'),
    ('Blancas con una torre de mas',    'rnbqkbn1/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQq - 0 1'),
    ('Blancas con una dama de mas',     'rnb1kbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1'),
    ('Negras con una dama de mas',      'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR b KQkq - 0 1'),
    ('Rey y dama contra rey',           '8/8/8/4k3/8/8/8/3QK3 w - - 0 1'),
    ('Rey y torre contra rey (negras)', '4k2r/8/8/8/8/8/8/4K3 w - - 0 1'),
    ('Mate en 1 de las blancas',        '6k1/4Rppp/8/8/8/8/5PPP/6K1 w - - 0 1'),
    ('Mate en 1 de las negras',         '6k1/5ppp/8/8/8/8/4rPPP/6K1 b - - 0 1'),
    ('Dama blanca en prise (tactica)',  'rnbqkbnr/pppp1ppp/8/4p3/3QP3/8/PPP2PPP/RNB1KBNR b KQkq - 0 3'),
]
PAREJA_CP = 100   # por debajo de esto, la posicion se lee como pareja

# Stockfish a la profundidad del etiquetado, en vista de las blancas.
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    ref_sf = [score_to_cp(sf.analyse(chess.Board(fen), limite_sf)['score'].white())
              for _, fen in REFERENCIA]

# La red, sin busqueda: valor al turno -> vista blancas -> centipeones (4.2).
ref_red = {}
for nombre, ev in evaluadores.items():
    ref_red[nombre] = []
    for _, fen in REFERENCIA:
        board = chess.Board(fen)
        ref_red[nombre].append(value_white(board, ev.evaluate_board(board)))

print(f"{'posicion':<33}{'Stockfish':>10}" + ''.join(f'{n:>17}' for n in evaluadores))
print(f"{'':<33}{'cp':>10}" + f"{'valor':>10}{'cp':>7}" * len(evaluadores))
print('-' * (43 + 17 * len(evaluadores)))
for k, (nombre_pos, _) in enumerate(REFERENCIA):
    cp_sf, mate = ref_sf[k]
    fila = f"{nombre_pos:<33}{(f'#{mate:+d}' if mate is not None else f'{cp_sf:+d}'):>10}"
    for nombre in evaluadores:
        v = ref_red[nombre][k]
        fila += f'{v:>+10.3f}{value_to_cp(v):>+7.0f}'
    print(fila)

# Coherencia: signo en las posiciones con ventaja clara, cerca de cero en las parejas.
coherencia = {}
print()
for nombre in evaluadores:
    valores = np.array(ref_red[nombre])
    assert np.all(np.abs(valores) <= 1.0)          # requerimiento 1.4
    claras = [(cp, v) for (cp, _), v in zip(ref_sf, valores) if abs(cp) >= PAREJA_CP]
    parejas = [v for (cp, _), v in zip(ref_sf, valores) if abs(cp) < PAREJA_CP]
    signo = sum(np.sign(cp) == np.sign(v) for cp, v in claras)
    cerca = sum(abs(value_to_cp(v)) < PAREJA_CP for v in parejas)
    coherencia[nombre] = (signo, len(claras), cerca, len(parejas))
    print(f'{nombre:<12} signo como Stockfish en {signo} de {len(claras)} posiciones con ventaja clara; '
          f'{cerca} de {len(parejas)} parejas quedan a menos de {PAREJA_CP} cp de cero')

## 7. Escalera de Elo, a 1 y 2 plies

La sección 8 de la 08, con un escalón más. Lo que no cambia, y es lo que hace
comparables los números:

- Stockfish con la fuerza acotada por `UCI_Elo` y **límite de tiempo** de 50 ms
  por jugada, no de profundidad;
- las **mismas 15 aperturas** del split de test, cada una con los dos colores;
- el Elo **ajustado contra todos los escalones a la vez** (`pooled_elo`), con ±
  un sigma.

Lo que cambia es el **escalón de 1900**. Cada motor tiene entonces dos Elo: el
ajustado contra los cuatro escalones, que es el número del apéndice, y el
ajustado contra los tres de la 08 (sin las partidas contra 1900), que es el que
se compara en igualdad de condiciones con los modelos de la memoria. En la 08
el Elo implícito subía con el escalón; si eso se repite, el de cuatro escalones
va a dar algo más alto.

**Requerimiento 2.4**, como en la 08: cada motor juega 120 partidas, y la tabla
de abajo da el porcentaje de victorias, tablas y derrotas y cuántas terminaron
por las reglas.

In [ ]:
from chessdl.engine.match import (
    MatchResult, estimated_seconds, opening_positions, play_match, pooled_elo,
    stockfish_at_elo,
)

APERTURAS = 15               # x2 partidas cada una
ESCALONES = [1320, 1500, 1700, 1900]
ESCALONES_08 = [1320, 1500, 1700]   # los de la 08, para comparar en igualdad
PROFUNDIDADES = [1, 2]       # la 3 no entra en el 1.7; ver el texto de arriba
TIEMPO_SF = 0.05             # segundos por jugada del rival

aperturas = opening_positions([fens[i] for i in orden], plies[orden],
                              count=APERTURAS, max_ply=16, seed=0)
print(f'{len(aperturas)} aperturas del split de test, {2*len(aperturas)} partidas por escalon\n')

# Las jugadas del rival tambien cuestan: una por cada jugada del motor.
print('costo estimado del torneo completo, por profundidad:')
for profundidad in (1, 2, 3):
    total = sum(estimated_seconds(APERTURAS, costo[(n, profundidad)] + TIEMPO_SF)
                * len(ESCALONES) for n in evaluadores)
    print(f'  profundidad {profundidad}: {total/60:6.1f} min'
          + ('  <- se corre' if profundidad in PROFUNDIDADES else ''))

In [ ]:
# Cada combinacion de red y profundidad es un motor distinto, con su propio Elo.
motores = {f'{nombre} d{d}': (ev, d) for nombre, ev in evaluadores.items() for d in PROFUNDIDADES}

torneos = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    for motor, (ev, profundidad) in motores.items():
        for elo in ESCALONES:
            stockfish_at_elo(sf, elo)
            clave = f'{motor} vs SF {elo}'
            torneos[clave] = play_match(
                # Limite de TIEMPO y no de profundidad: `UCI_Elo` esta calibrado
                # para busquedas con control de tiempo, y fijarle la profundidad
                # a mano pisa el mecanismo con el que Stockfish se debilita. Con
                # `depth=6` el rival juega debilitado, pero su Elo no es el numero
                # de la etiqueta -- que para la memoria es peor que no tenerlo.
                ev, sf, aperturas, chess.engine.Limit(time=TIEMPO_SF),
                depth=profundidad, label=clave, progress=True,
            )
            print(torneos[clave].summary()); print()


def elo_corto(ajuste):
    """El Elo ajustado para una columna de tabla; si no esta acotado, de que lado."""
    if ajuste.bounded:
        return ajuste.describe()
    return f'> {max(ESCALONES)}' if ajuste.rating > 0 else f'< {min(ESCALONES)}'


# Un Elo por motor, ajustado contra todos los escalones a la vez -- no leido de uno.
elos = {motor: pooled_elo({e: torneos[f'{motor} vs SF {e}'] for e in ESCALONES})
        for motor in motores}
# Y el mismo ajuste sin el escalon nuevo: la escalera exacta de la 08.
elos_08 = {motor: pooled_elo({e: torneos[f'{motor} vs SF {e}'] for e in ESCALONES_08})
           for motor in motores}

print('ELO AJUSTADO CONTRA TODOS LOS ESCALONES A LA VEZ')
print('-' * 98)
print(f"{'':<16}" + ''.join(f'{f"vs {e}":>10}' for e in ESCALONES)
      + f"{'Elo (4 escalones)':>20}{'Elo (los 3 de la 08)':>22}")
for motor, ajuste in elos.items():
    tasas = ''.join(f'{torneos[f"{motor} vs SF {e}"].score_rate:>10.3f}' for e in ESCALONES)
    print(f'{motor:<16}{tasas}{elo_corto(ajuste):>20}{elo_corto(elos_08[motor]):>22}')
print('\n(puntos por partida contra cada escalon; el Elo usa las '
      f'{next(iter(elos.values())).games} partidas de cada motor, +- un sigma)')
# Requerimiento 2.4: las 90 partidas de cada motor contra Stockfish a Elo bajo.
escalera = {motor: MatchResult([g for e in ESCALONES for g in torneos[f'{motor} vs SF {e}'].games],
                               label=f'{motor} vs SF {min(ESCALONES)}-{max(ESCALONES)}')
             for motor in motores}
print()
print(f'REQUERIMIENTO 2.4: TODAS LAS PARTIDAS DE LA ESCALERA')
print('-' * 72)
print(f"{'':<16}{'partidas':>9}{'victorias':>11}{'tablas':>9}{'derrotas':>10}{'por las reglas':>17}")
for motor, t in escalera.items():
    v, e, d = t.percentages
    print(f'{motor:<16}{len(t.games):>9}{v:>11.0%}{e:>9.0%}{d:>10.0%}'
          f'{f"{len(t.games) - t.unfinished} de {len(t.games)}":>17}')

## 8. Contra Stockfish a un ply

La sección 9 de la 08: Stockfish a fuerza completa pero limitado a un ply, contra
el motor también a un ply. Misma búsqueda de los dos lados, así que lo único
distinto es la función de evaluación. Ese rival no tiene un Elo calibrado: el
resultado compara evaluaciones y no se traduce a una escala.

In [ ]:
from chessdl.engine.match import stockfish_full_strength

torneos_1ply = {}
with chess.engine.SimpleEngine.popen_uci('./bin/stockfish') as sf:
    stockfish_full_strength(sf)
    for nombre, ev in evaluadores.items():
        clave = f'{nombre} d1 vs SF d1'
        torneos_1ply[clave] = play_match(
            ev, sf, aperturas, chess.engine.Limit(depth=1),
            depth=1, label=clave, progress=True,
        )
        print(torneos_1ply[clave].summary()); print()

## 9. Comparación con los modelos de la memoria

Los números de la ResNet y el Transformer de la memoria se toman de la corrida de
la [08](../08_motor_y_partidas.ipynb), y la celda dice de qué sección sale cada
uno. Para los modelos del apéndice se usa, en la tabla, el Elo ajustado contra
**los mismos tres escalones** que la 08, que es la comparación en igualdad de
condiciones; el de cuatro escalones está en la sección 7.

In [ ]:
from chessdl.engine.match import PooledElo
from chessdl.training.metrics import spearman

# Los dos modelos de la memoria, de la corrida de la 08. No se vuelven a medir.
MEMORIA = {
    'ResNet memoria': dict(
        rmse=0.2511,                                    # 08, seccion 5.2
        acpl={1: 176.8, 2: 87.4}, graves={1: 0.210, 2: 0.055},   # seccion 7
        p99={1: 23, 2: 388},                            # seccion 6, ms
        elo={1: PooledElo(1124, 56, 90), 2: PooledElo(1534, 40, 90)},   # seccion 8
        sf1ply=0.283),                                  # seccion 9
    'Transformer memoria': dict(
        rmse=0.2531,
        acpl={1: 257.1, 2: 97.9}, graves={1: 0.342, 2: 0.080},
        p99={1: 13, 2: 583},
        elo={1: PooledElo(916, 90, 90), 2: PooledElo(1373, 41, 90)},
        sf1ply=0.250),
}

# Los del apendice, medidos en esta notebook, en la misma forma.
APENDICE = {
    nombre: dict(
        rmse=metricas[nombre][1].rmse,
        acpl={d: calidades[f'{nombre} d{d}'].acpl for d in PROFUNDIDADES},
        graves={d: calidades[f'{nombre} d{d}'].blunder_rate() for d in PROFUNDIDADES},
        p99={d: float(np.percentile(tiempos_1_7[(nombre, d)], 99)) for d in PROFUNDIDADES},
        elo={d: elos_08[f'{nombre} d{d}'] for d in PROFUNDIDADES},
        sf1ply=torneos_1ply[f'{nombre} d1 vs SF d1'].score_rate)
    for nombre in evaluadores
}
TODOS = {**APENDICE, **MEMORIA}

print(f"{'modelo':<21}{'RMSE test':>10}{'ACPL d1':>9}{'ACPL d2':>9}{'>300 d2':>9}"
      f"{'Elo d1':>13}{'Elo d2':>13}{'vs SF 1 ply':>13}")
print('-' * 97)
for nombre, m in sorted(TODOS.items(), key=lambda kv: kv[1]['rmse']):
    print(f"{nombre:<21}{m['rmse']:>10.4f}{m['acpl'][1]:>9.1f}{m['acpl'][2]:>9.1f}"
          f"{m['graves'][2]:>9.1%}{elo_corto(m['elo'][1]):>13}{elo_corto(m['elo'][2]):>13}"
          f"{m['sf1ply']:>13.3f}")
print('\n(apendice: Elo contra los mismos 3 escalones que la 08; memoria: valores de la 08)')

# Cuanto acompaña el RMSE a la fuerza. Con seis modelos el rho es orientativo:
# importa el signo, y si se sostiene en las dos profundidades.
print()
x = np.array([m['rmse'] for m in TODOS.values()])
for etiqueta, y, esperado in (
        ('ACPL a 1 ply', [m['acpl'][1] for m in TODOS.values()], +1),
        ('ACPL a 2 plies', [m['acpl'][2] for m in TODOS.values()], +1),
        ('Elo a 1 ply', [m['elo'][1].rating for m in TODOS.values()], -1),
        ('Elo a 2 plies', [m['elo'][2].rating for m in TODOS.values()], -1)):
    rho = spearman(x, np.array(y, dtype=float))
    lectura = ('sin señal' if abs(rho) < 0.3 else
               'el RMSE acompaña' if np.sign(rho) == esperado else 'va AL REVES')
    print(f'RMSE vs {etiqueta:<15} rho de Spearman {rho:+.2f}   {lectura}')

### 9.1 El Elo de todos los modelos, a 1 y a 2 plies

Un gráfico por profundidad, con la misma escala horizontal en los dos para que
se puedan comparar entre sí. Cada punto es el Elo ajustado y la barra, un sigma;
las líneas punteadas son los escalones de Stockfish de la escalera.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from chessdl import viz

viz.apply_style()

ETIQUETA = {'ResNet memoria': 'ResNet (memoria)',
            'Transformer memoria': 'Transformer (memoria)',
            '30M-grande': '30M-grande (16 bloques)'}

# La misma escala horizontal en los dos graficos.
acotados = [m['elo'][d] for m in TODOS.values() for d in PROFUNDIDADES
            if m['elo'][d].bounded]
X_MIN = min([a.rating - a.sigma for a in acotados] + ESCALONES_08) - 120
X_MAX = max([a.rating + a.sigma for a in acotados] + ESCALONES_08) + 220


def grafico_elo(d):
    """Los seis modelos a una profundidad, del mas fuerte (arriba) al mas debil."""
    def clave(nombre):
        a = TODOS[nombre]['elo'][d]
        return a.rating if a.bounded else (-1e9 if a.rating < 0 else 1e9)

    filas = sorted(TODOS, key=clave)
    fig, eje = plt.subplots(figsize=(8.6, 0.55 * len(filas) + 1.9))

    for e in ESCALONES_08:
        eje.axvline(e, color=viz.AXIS, linewidth=1, linestyle=(0, (3, 3)), zorder=0)
        eje.text(e, len(filas) - 0.45, f'SF {e}', ha='center', va='bottom',
                 fontsize=7.5, color=viz.INK_MUTED)

    for i, nombre in enumerate(filas):
        a = TODOS[nombre]['elo'][d]
        color = viz.SERIES[1] if nombre in MEMORIA else viz.SERIES[0]
        if a.bounded:
            eje.errorbar(a.rating, i, xerr=a.sigma, fmt='o', markersize=8, color=color,
                         capsize=3, elinewidth=1.4, markeredgecolor=viz.SURFACE,
                         markeredgewidth=1.5, zorder=3)
            x_texto, texto = a.rating + a.sigma, f'{a.rating:.0f} ± {a.sigma:.0f}'
        else:
            # Barrido: no hay numero. La flecha dice de que lado queda.
            x_texto = X_MIN + 40 if a.rating < 0 else X_MAX - 40
            eje.plot(x_texto, i, marker='<' if a.rating < 0 else '>', markersize=9,
                     color=color, linestyle='none', zorder=3)
            texto = elo_corto(a)
        eje.annotate(texto, (x_texto, i), xytext=(8, 0), textcoords='offset points',
                     va='center', fontsize=8, color=viz.INK_SECONDARY)

    eje.set_yticks(range(len(filas)), [ETIQUETA.get(n, n) for n in filas])
    eje.set_ylim(-0.6, len(filas) - 0.05)
    eje.set_xlim(X_MIN, X_MAX)
    viz.label_axes(eje, f'Elo a {d} {"ply" if d == 1 else "plies"}',
                   'Elo ajustado (escala UCI_Elo de Stockfish)', '',
                   note='punto: Elo ajustado contra la escalera de la 08; barra: ± un sigma')
    eje.grid(axis='x', visible=True)
    eje.grid(axis='y', visible=False)
    eje.legend(handles=[
        Line2D([], [], marker='o', linestyle='none', markersize=8, color=viz.SERIES[0],
               label='apéndice (medido acá)'),
        Line2D([], [], marker='o', linestyle='none', markersize=8, color=viz.SERIES[1],
               label='memoria (de la 08)')],
        # Debajo del eje, fuera del area de datos: adentro choca con la etiqueta
        # de alguna fila segun donde caigan los modelos.
        loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=2)
    fig.tight_layout()
    plt.show()


for d in PROFUNDIDADES:
    grafico_elo(d)

## 10. Qué sigue

La **notebook 4** cierra el apéndice con el tablero de la
[09](../09_jugar_contra_el_motor.ipynb) apuntado a los modelos más fuertes de
acá, para jugarles y ver a mano lo que estas tablas dicen en números.

> **La salvedad de todo el apéndice.** Los modelos del apéndice se entrenaron con
> otras etiquetas y otra distribución de posiciones, y con un presupuesto de
> épocas distinto. Lo que se compara en esta notebook está medido con la misma
> vara —el test del proyecto, las mismas posiciones, la misma escalera—, pero
> la diferencia no es sólo de volumen: también cambió la fuente de los datos.

**Próximo paso:** [`lichess_4_jugar.ipynb`](lichess_4_jugar.ipynb), el tablero
para jugarles.